# Figure 5 and Supplementary Figures 9–12: long-range concordance in mouse cortex

This notebook reproduces, from the per-cell BAM files of the 1,577 mouse cortex nuclei:

| Figure | What | Made in |
|---|---|---|
| Fig. 5a,b | concordance by distance; GCH against HCG across 10-Mb regions | Steps 3, 5 |
| Fig. 5c,d | concordance in each cell, and by cell type | Steps 3, 5 |
| Fig. 5e–g | concordance by chromatin class, chromHMM state and along the genome | Steps 4, 5 |
| Supplementary Fig. 9 | the model (schematic; no data) | Step 5 |
| Supplementary Fig. 10 | informative sites per read pair | Step 2 |
| Supplementary Fig. 11 | Fig. 5a–c under three non-linked controls | Steps 3, 5 |
| Supplementary Fig. 12 | Fig. 5a,b and e,f under the other near-copy rule | Steps 3–5 |

Each command is skipped when its outputs already exist, so the notebook can be re-run, or resumed after an interruption; set `DRY_RUN = True` to print the commands without running them. See `README.md` (this folder) for installation, data download and compute requirements. The last section compares key numbers with the manuscript.

## Configuration

Paths come from `paths.yaml` at the repository root; edit that, not this notebook. The per-cell BAM files are read where the pipeline wrote them (`<work_root>/mouse_cortex/04.alignment_snakemake/`), meth_concord is the checkout at `<software_root>/meth_concord`, and this analysis keeps the public data (`annotations/`, made by `download_public_data.sh`) and its results (`results/mouse_cortex/`) under `<work_root>/concordance/`.

In [ ]:
# --- environment ---------------------------------------------------------------
# This notebook was written against the `meth_concord` conda environment, made
# as in "1. Install" of this folder's README.md.
# See env/README.md for which notebook uses which environment.
_ENV_EXPECTED = "meth_concord"
import os as _os, sys as _sys
if _os.path.basename(_sys.prefix) != _ENV_EXPECTED:
    print(f"[env] expected the '{_ENV_EXPECTED}' environment, running in "
          f"'{_os.path.basename(_sys.prefix)}' ({_sys.prefix}). See env/README.md.",
          file=_sys.stderr)

# --- repo paths: edit paths.yaml at the repository root, not this notebook ---
if "REPO_ROOT" not in globals():
    _r = _os.getcwd()
    while _r != "/" and not _os.path.exists(_os.path.join(_r, "paths.py")):
        _r = _os.path.dirname(_r)
    assert _os.path.exists(_os.path.join(_r, "paths.py")), \
        "paths.py not found above the working directory -- start Jupyter inside the repository"
    REPO_ROOT = _r
_sys.path.insert(0, REPO_ROOT)
from paths import SOFTWARE, WORK

import sys
from pathlib import Path

HERE = Path(REPO_ROOT) / "main_figures" / "fig5_fig6"   # this folder
METH_CONCORD = Path(SOFTWARE) / "meth_concord"          # software/README.md
BAMS = Path(WORK)                   # the sample sheet's BAM paths start here
DATA = Path(WORK) / "concordance"   # the public data and the results
ANNOT = DATA / "annotations"        # download_public_data.sh ANNOT mm10
OUT = DATA / "results" / "mouse_cortex"
THREADS = 10
DRY_RUN = False                     # True: print the commands only
# chromHMM's bivalent-promoter states: "polycomb", within the Polycomb group
# (six groups, as in Fig. 5f and Supplementary Fig. 12d), or "own", a group of
# their own
CHROMHMM_BIVALENT = "polycomb"

In [ ]:
sys.path.insert(0, str(HERE))
import pandas as pd
from repro import compare, run, show

PY = sys.executable
S = METH_CONCORD / "scripts"
MM10 = ANNOT / "mm10"
BLACKLIST = MM10 / "mm10-blacklist.v2.bed"
LOGS, FIG = OUT / "logs", OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

# The sample sheet (cell, BAM, cell type, batches, colour), with the BAM
# paths made absolute under BAMS.
sheet = pd.read_csv(HERE / "samples/mouse_cortex.samples.tsv", sep="\t")
sheet["bam_path"] = [str(BAMS / p) for p in sheet["bam_path"]]
SAMPLES = OUT / "mouse_cortex.samples.tsv"
sheet.to_csv(SAMPLES, sep="\t", index=False)
missing = [p for p in sheet["bam_path"] if not Path(p).exists()]
print(f"{len(sheet):,} cells:", sheet["group"].value_counts().to_dict())
print(f"{len(missing):,} BAM files missing" + (f", e.g. {missing[0]}" if missing else ""))
if not (S / "figure5_cortex.py").exists():
    print(f"meth_concord not found at {METH_CONCORD}: see software/README.md")

## Step 1. End posteriors

The pipeline's steps 1–4, once per context (GCH, accessibility; HCG, methylation):

1. the context's cytosines on the autosomes, from the `seqkit` BED (GCG excluded);
2. every read pair with mapping quality ≥30 at both mates, duplicates removed, as two ends with *k* methylated of *n* informative sites each (5 bp trimmed from each read end, sites read on the read's bisulfite-conversion strand);
3. a two-component beta-binomial mixture fitted to the ends of all cells (up to 10⁷ ends, five restarts, seed 42);
4. each end's posterior probability of the higher component, open (GCH) or methylated (HCG).

Outputs: `pipeline/context_<ctx>/02_read_pair_counts` (Step 2) and `pipeline/context_<ctx>/04_posteriors` (everything else). This is the longest step.

In [ ]:
PIPE = OUT / "pipeline"
for ctx in ("GCH", "HCG"):
    bed = MM10 / f"{ctx}_motif_seqkit.mm10.bed.gz"
    run([PY, "-m", "nome_hic_concordance",
         "--samples", SAMPLES, "--output-dir", PIPE,
         "--contexts", ctx, "--context-bed", f"{ctx}:{bed}",
         "--reference", MM10 / "mm10.2bit", "--blacklist-bed", BLACKLIST,
         "--exclude-chrom", "chrM,chrX,chrY",
         "--steps", "1,2,3,4", "--n-threads", THREADS],
        outputs=[PIPE / f"context_{ctx}" / "04_posteriors"],
        inputs=[SAMPLES, bed, MM10 / "mm10.2bit", BLACKLIST],
        log=LOGS / f"pipeline_{ctx}.log", dry_run=DRY_RUN)

POSTERIORS = ["--posteriors", f"GCH={PIPE / 'context_GCH' / '04_posteriors'}",
              "--posteriors", f"HCG={PIPE / 'context_HCG' / '04_posteriors'}"]

## Step 2. Supplementary Fig. 10: informative sites per read pair

The share of read pairs with informative GCH, HCG or both sites at neither, one or both ends, over all read pairs (a) and over pairs more than 1 kb (b) and 20 kb (c) apart.

In [ ]:
RPS = {"all": [], "gt1kb": ["--min-distance", "1000"], "gt20kb": ["--min-distance", "20000"]}
for name, extra in RPS.items():
    run([PY, S / "read_pair_stats.py",
         "--gch-read-pairs-dir", PIPE / "context_GCH" / "02_read_pair_counts",
         "--hcg-read-pairs-dir", PIPE / "context_HCG" / "02_read_pair_counts",
         *extra, "--out", FIG / f"FigureS10_{name}.pdf", "--n-threads", THREADS],
        outputs=[FIG / f"FigureS10_{name}.pdf"],
        log=LOGS / f"read_pair_stats_{name}.log", dry_run=DRY_RUN)

## Step 3. Concordance by distance, under three non-linked controls

Each linked pair (the two ends of one read pair) is compared with a non-linked pair that keeps its anchor end and replaces its partner (Supplementary Fig. 9c):

| Control | The stand-in | Used in |
|---|---|---|
| 1 | another read pair's end of the same cell, in the partner's 250-kb bin and distance bin; near-copies of the contact refused (within 1 kb of the partner, or its mate within 1 kb of the anchor) | Fig. 5c,d; Suppl. Figs. 11, 12a,b |
| 2 | the same, with near-copies allowed | Fig. 5a,b; Suppl. Fig. 11 |
| 3 | an end within 1 kb of the partner's position, from another cell of the same cell type and batch | Suppl. Fig. 11 |

Each end's posterior is centred on its own cell's mean. Each run also writes the per-cell test (`end_posterior_percell.tsv`) and the 10-Mb regions (`end_posterior_regions*.tsv`). Control 3 builds each stratum's donor ends in turn; `--pool-max-cells` caps how many cells' donors are held at once (300 cells, roughly 9–12 GB).

In [ ]:
DIST = OUT / "distance"
GROUP_ORDER = "IT-L2/3,IT-L4,IT-L5/6,CT-L6,Inhib,ODC,OPC,ASC,MGC,PC/EC"
common = [*POSTERIORS, "--samples", SAMPLES, "--group-column", "group",
          "--exclude-chrom", "chrM,chrX,chrY", "--exclude-bed", BLACKLIST,
          "--min-sites-gch", "1", "--min-sites-hcg", "1",
          "--delta-arm", "matched", "--cell-center", "on", "--seed", "42",
          "--color-column", "color", "--group-order", GROUP_ORDER,
          "--plot-format", "pdf", "--n-threads", THREADS]
CONTROLS = {
    "control1_same_cell_near_copies_refused":
        ["--null-scope", "cell", "--null-type", "link-shuffle",
         "--null-match-column", "exp_batch,lib_batch", "--null-exclude", "1000"],
    "control2_same_cell_near_copies_kept":
        ["--null-scope", "cell", "--null-type", "link-shuffle",
         "--null-match-column", "exp_batch,lib_batch"],
    "control3_same_place_other_cells":
        ["--null-scope", "pooled", "--null-type", "locus-matched",
         "--null-window", "1000", "--null-match-column", "group,exp_batch",
         "--pool-max-cells", "300"],
}
for name, null in CONTROLS.items():
    run([PY, S / "end_posterior_distance_correlation.py", *common, *null,
         "--output-dir", DIST / name],
        outputs=[DIST / name / "end_posterior_percell.tsv",
                 DIST / name / "end_posterior_regions_summary.tsv"],
        log=LOGS / f"distance_{name}.log", dry_run=DRY_RUN)
C1, C2, C3 = (DIST / n for n in CONTROLS)

## Step 4. Concordance by chromatin class and along the genome

Each contact is classed by the annotation at its two ends: A/B compartment (called per cell type from these cells' own contacts), replication timing (4DN Repli-seq of neural precursor cells), common PMD/HMD and the universal chromHMM states of mm10 in six groups (`CHROMHMM_BIVALENT = "polycomb"` keeps the bivalent-promoter states in the Polycomb group, as in the manuscript; `"own"` gives them a group of their own and its runs a folder of their own). ΔC compares the contacts' concordance with reshuffled links from the same cell, class and 250-kb bin. Run once with near-copies refused (Fig. 5e–g) and once with them allowed (Supplementary Fig. 12c,d). The first run downloads the chromHMM model, GENCODE vM25 and the UCSC CpG islands into `ANNOT/cache`.

In [ ]:
CLS = OUT / ("classes" if CHROMHMM_BIVALENT == "polycomb" else
             f"classes_bivalent_{CHROMHMM_BIVALENT}")
RT = MM10 / "repliseq"
EARLY = ["F121-9-CASTx129_NPC.repliseq.early.mm10.4DNFINCFK1F9",
         "Cas129-CD1-129_CAST.repliseq.early.rep1.4DNFIC5KD21S",
         "Cas129-CD1-129_CAST.repliseq.early.rep2.4DNFI2D6RJL8"]
LATE = ["F121-9-CASTx129_NPC.repliseq.late.mm10.4DNFIBA95HHW",
        "Cas129-CD1-129_CAST.repliseq.late.rep1.4DNFIDSG5BWJ",
        "Cas129-CD1-129_CAST.repliseq.late.rep2.4DNFIDJYEKHB"]
rt = [a for f in EARLY for a in ("--rt-early", RT / f"{f}.bw")]
rt += [a for f in LATE for a in ("--rt-late", RT / f"{f}.bw")]
common_cls = [*POSTERIORS, "--samples", SAMPLES,
              "--batch-column", "exp_batch,lib_batch", "--genome", "mm10",
              "--systems", "ab,rt,chromhmm,pmd", *rt,
              "--pmd", MM10 / "PMD_coordinates_mm10.bed.gz",
              "--chromhmm-bivalent", CHROMHMM_BIVALENT,
              "--null-scope", "cell", "--bins-per-decade", "8",
              "--max-widen", "0", "--position-bin", "250000",
              "--min-sites", "GCH=1,HCG=1", "--exclude-chrom", "chrM,chrX,chrY",
              "--exclude-bed", BLACKLIST, "--cache-dir", ANNOT / "cache",
              "--n-threads", THREADS]
RUNS = {"near_copies_refused": ["--null-exclude", "1000"], "near_copies_kept": []}
for name, extra in RUNS.items():
    run([PY, S / "annotation_class_concordance.py", *common_cls, *extra,
         "--output-dir", CLS / name],
        outputs=[CLS / name / "summary.tsv", CLS / name / "locus_hotspots.tsv.gz"],
        inputs=[*rt[1::2], MM10 / "PMD_coordinates_mm10.bed.gz", BLACKLIST],
        log=LOGS / f"classes_{name}.log", dry_run=DRY_RUN)

## Step 5. The figures

`figure5_cortex.py` draws Fig. 5 (a,b from control 2; c,d from control 1; e–g from the near-copies-refused class run) and, with `--supplementary`, Supplementary Fig. 12 (a,b from control 1; c,d from the near-copies-allowed class run). `figure_s_null_controls.py` draws Supplementary Fig. 11 and `figure_s9_concordance_model.py` the schematic of Supplementary Fig. 9. Figures are redrawn on every run (seconds).

In [ ]:
run([PY, S / "figure5_cortex.py", "--distance", C1, "--distance-ab", C2,
     "--classes", CLS / "near_copies_refused",
     "--classes-supp", CLS / "near_copies_kept",
     "--output", FIG / "Figure5.pdf", "--supplementary", FIG / "FigureS12.pdf"],
    outputs=[FIG / "Figure5.pdf"], log=LOGS / "figure5.log",
    dry_run=DRY_RUN, force=True)
run([PY, S / "figure_s_null_controls.py", "--run", f"1={C1}", "--run", f"2={C2}",
     "--run", f"3={C3}", "--output", FIG / "FigureS11.pdf"],
    outputs=[FIG / "FigureS11.pdf"], log=LOGS / "figureS11.log",
    dry_run=DRY_RUN, force=True)
run([PY, S / "figure_s9_concordance_model.py", "--output", FIG / "FigureS9.pdf"],
    outputs=[FIG / "FigureS9.pdf"], log=LOGS / "figureS9.log",
    dry_run=DRY_RUN, force=True)

In [ ]:
show(FIG / "Figure5.pdf")

In [ ]:
for name in ("FigureS9", "FigureS10_all", "FigureS10_gt1kb", "FigureS10_gt20kb",
             "FigureS11", "FigureS12"):
    show(FIG / f"{name}.pdf", dpi=80)

## Step 6. Checks against the manuscript

The runs are seeded, so the numbers below should match the manuscript's. With `CHROMHMM_BIVALENT = "own"` the heterochromatin–heterochromatin ΔC is checked against that grouping's 2.79 (the others do not change).

In [ ]:
import importlib.util


def load(name):
    spec = importlib.util.spec_from_file_location(name, S / f"{name}.py")
    m = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m)
    return m


def checks():
    f5 = load("figure5_cortex")
    pc = pd.read_csv(C1 / "end_posterior_percell.tsv", sep="\t").dropna(subset=["delta"])
    pc3 = pd.read_csv(C3 / "end_posterior_percell.tsv", sep="\t").dropna(subset=["delta"])
    reg = {k: pd.read_csv(d / "end_posterior_regions_summary.tsv", sep="\t").iloc[0]
           for k, d in (("1", C1), ("2", C2), ("3", C3))}
    summ = pd.read_csv(CLS / "near_copies_refused" / "summary.tsv", sep="\t")
    hot = pd.read_csv(CLS / "near_copies_refused" / "locus_hotspots.tsv.gz", sep="\t")
    hot = hot[(hot["band"] == f5.ALL_BAND) & hot["tested"].astype(bool)]

    def pct_pos(t, mark):
        return 100 * (t.loc[t["mark"] == mark, "delta"] > 0).mean()

    def pp(system, a, b, mark, stat="delta"):
        return 100 * float(f5._pairs(summ, mark, [(system, a, b, "")])[stat].iloc[0])

    def n_hot(mark, col):
        t = hot[hot["context"] == mark]
        return int(t[col].astype(bool).sum()) if col else len(t)

    return compare([
        ("Fig. 5b: regions, Pearson r", "0.52", reg["2"]["pearson_r"]),
        ("Fig. 5c: GCH cells with Δr > 0 (%)", "99.8", pct_pos(pc, "GCH")),
        ("Fig. 5c: HCG cells with Δr > 0 (%)", "72.9", pct_pos(pc, "HCG")),
        ("Fig. 5c: median per-cell GCH Δr", "0.061", pc.loc[pc["mark"] == "GCH", "delta"].median()),
        ("Fig. 5c: median per-cell HCG Δr", "0.022", pc.loc[pc["mark"] == "HCG", "delta"].median()),
        ("Fig. 5e: GCH ΔC, B–B (pp)", "1.56", pp("ab", "B", "B", "GCH")),
        ("Fig. 5e: GCH ΔC, A–A (pp)", "0.99", pp("ab", "A", "A", "GCH")),
        ("Fig. 5f: GCH ΔC × w, enhancer–promoter (pp)", "0.52",
         pp("chromhmm", "enhancer", "promoter", "GCH", "signed_rel")),
        ("Fig. 5f: GCH ΔC, heterochromatin–heterochromatin (pp)",
         "2.86" if CHROMHMM_BIVALENT == "polycomb" else "2.79",
         pp("chromhmm", "heterochromatin", "heterochromatin", "GCH")),
        ("Fig. 5g: GCH bins tested", "2393", n_hot("GCH", None)),
        ("Fig. 5g: GCH hot bins", "146", n_hot("GCH", "hot")),
        ("Fig. 5g: HCG bins tested", "659", n_hot("HCG", None)),
        ("Fig. 5g: HCG hot bins", "23", n_hot("HCG", "hot")),
        ("Suppl. Fig. 11: regions, r (control 3)", "0.36", reg["3"]["pearson_r"]),
        ("Suppl. Fig. 11: HCG cells with Δr > 0, control 3 (%)", "73.0", pct_pos(pc3, "HCG")),
        ("Suppl. Fig. 12b: regions, Pearson r", "0.51", reg["1"]["pearson_r"]),
    ])


print("dry run: nothing to check yet") if DRY_RUN else checks()